# Einstein Summation and Index Notation

Interactive examples demonstrating Einstein convention, index notation, and `np.einsum`/`torch.einsum`.

**Topics covered:**
1. From explicit Σ to Einstein convention
2. Free indices vs dummy indices
3. einsum for all tensor operations
4. Batch operations and multi-head attention
5. Kronecker delta and identity
6. Performance and optimization

In [ ]:
import numpy as np
np.random.seed(42)

## 1. From Σ to Einstein Convention

Einstein's rule: **an index appearing twice in a term is summed over**.

| Explicit Σ | Einstein | einsum |
|---|---|---|
| Σᵢ aᵢbᵢ | aᵢbᵢ | `'i,i->'` |
| Σₖ AᵢₖBₖⱼ | AᵢₖBₖⱼ | `'ik,kj->ij'` |
| Σᵢ Aᵢᵢ | Aᵢᵢ | `'ii->'` |

In [ ]:
# From explicit loops to einsum

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# === DOT PRODUCT: c = Σᵢ aᵢbᵢ ===
# Method 1: Explicit loop
dot_loop = 0
for i in range(len(a)):
    dot_loop += a[i] * b[i]

# Method 2: NumPy
dot_numpy = np.dot(a, b)

# Method 3: Einstein (einsum)
dot_einsum = np.einsum('i,i->', a, b)

print('=== Dot Product: c = Σᵢ aᵢbᵢ = aᵢbᵢ ===')
print(f'  a = {a}')
print(f'  b = {b}')
print(f'  Loop:   {dot_loop}')
print(f'  np.dot: {dot_numpy}')
print(f'  einsum: {dot_einsum}')
print(f'  All equal: {dot_loop == dot_numpy == dot_einsum}')

# === OUTER PRODUCT: Mᵢⱼ = aᵢbⱼ (no repeated index → no sum) ===
outer_einsum = np.einsum('i,j->ij', a, b)
outer_numpy = np.outer(a, b)

print(f'\n=== Outer Product: Mᵢⱼ = aᵢbⱼ ===')
print(f'  einsum result:\n{outer_einsum}')
print(f'  np.outer match: {np.array_equal(outer_einsum, outer_numpy)}')

# === KEY DIFFERENCE: dot vs element-wise ===
elem_wise = np.einsum('i,i->i', a, b)  # keep i in output → no sum!
print(f'\n=== Dot vs Element-wise ===')
print(f'  "i,i->"  (dot):          {np.einsum("i,i->", a, b)}   (i summed → scalar)')
print(f'  "i,i->i" (element-wise): {np.einsum("i,i->i", a, b)}  (i kept → vector)')

## 2. Matrix Operations via einsum

Every standard matrix operation has an einsum equivalent:

| Operation | einsum | Standard |
|---|---|---|
| Matmul | `'ik,kj->ij'` | `A @ B` |
| Mat-vec | `'ij,j->i'` | `A @ x` |
| Transpose | `'ij->ji'` | `A.T` |
| Trace | `'ii->'` | `np.trace(A)` |
| Diagonal | `'ii->i'` | `np.diag(A)` |
| Frobenius | `'ij,ij->'` | `np.sum(A*B)` |

In [ ]:
# All matrix operations via einsum
np.random.seed(42)

A = np.array([[1, 2, 3],
              [4, 5, 6]])
B = np.array([[7, 8],
              [9, 10],
              [11, 12]])
x = np.array([1, 2, 3])

print('=== Matrix Multiply: Cᵢⱼ = AᵢₖBₖⱼ ===')
C_einsum = np.einsum('ik,kj->ij', A, B)
C_matmul = A @ B
print(f'  einsum:\n{C_einsum}')
print(f'  A @ B match: {np.array_equal(C_einsum, C_matmul)}')

print(f'\n=== Matrix-Vector: cᵢ = Aᵢⱼxⱼ ===')
c_einsum = np.einsum('ij,j->i', A, x)
c_matmul = A @ x
print(f'  einsum: {c_einsum}')
print(f'  A @ x match: {np.array_equal(c_einsum, c_matmul)}')

print(f'\n=== Transpose: Bⱼᵢ = Aᵢⱼ ===')
AT_einsum = np.einsum('ij->ji', A)
print(f'  einsum:\n{AT_einsum}')
print(f'  A.T match: {np.array_equal(AT_einsum, A.T)}')

# Square matrix for trace and diagonal
S = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print(f'\n=== Trace: Aᵢᵢ (sum of diagonal) ===')
tr_einsum = np.einsum('ii->', S)
print(f'  einsum: {tr_einsum}')
print(f'  np.trace: {np.trace(S)}')

print(f'\n=== Diagonal: dᵢ = Aᵢᵢ ===')
diag_einsum = np.einsum('ii->i', S)
print(f'  einsum: {diag_einsum}')
print(f'  np.diag: {np.diag(S)}')

print(f'\n=== Frobenius Inner Product: AᵢⱼBᵢⱼ ===')
F_einsum = np.einsum('ij,ij->', S, S)
F_manual = np.sum(S * S)
print(f'  einsum (‖S‖²_F): {F_einsum}')
print(f'  np.sum(S*S):     {F_manual}')

## 3. Free vs Dummy Indices

The **output string** determines which indices are free (kept) vs dummy (summed):

```
'ij,jk->ik':  j is dummy (summed), i,k are free
'ij,jk->ijk': ALL are free (outer-product-like, no contraction)
'ij,jk->':    ALL are dummy (scalar result)
```

In [ ]:
# Demonstrating how output indices control the computation

A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# Same inputs, different output indices → different results
r1 = np.einsum('ij,jk->ik', A, B)   # Matrix multiply (sum over j)
r2 = np.einsum('ij,jk->ijk', A, B)  # No contraction (3D tensor)
r3 = np.einsum('ij,jk->', A, B)     # Sum everything (scalar)
r4 = np.einsum('ij,jk->j', A, B)    # Sum over i,k, keep j

print('Same inputs A(2×2), B(2×2), different output specs:')
print(f'\n"ij,jk->ik" (matmul):     shape {r1.shape}')
print(r1)
print(f'\n"ij,jk->ijk" (no contract): shape {r2.shape}')
print(r2)
print(f'\n"ij,jk->"  (total sum):    shape {r3.shape}, value = {r3}')
print(f'\n"ij,jk->j" (keep j only):  shape {r4.shape}, value = {r4}')

print('\n→ The output string is what makes einsum so powerful.')
print('→ It lets you express ANY contraction pattern in one call.')

## 4. Batch Operations

einsum handles batch dimensions effortlessly — just add a batch index:

| Operation | Single | Batched |
|---|---|---|
| Matmul | `'ik,kj->ij'` | `'bik,bkj->bij'` |
| Dot product | `'i,i->'` | `'bi,bi->b'` |
| Quadratic form | `'i,ij,j->'` | `'bi,ij,bj->b'` |

In [ ]:
# Batch operations — the killer feature of einsum
np.random.seed(42)

batch_size = 3
A_batch = np.random.randn(batch_size, 2, 4)  # 3 matrices, each 2×4
B_batch = np.random.randn(batch_size, 4, 3)  # 3 matrices, each 4×3

# Batch matrix multiply
C_batch = np.einsum('bik,bkj->bij', A_batch, B_batch)

print(f'Batch matmul: A({A_batch.shape}) × B({B_batch.shape}) = C({C_batch.shape})')

# Verify: same as looping
for b in range(batch_size):
    C_manual = A_batch[b] @ B_batch[b]
    match = np.allclose(C_batch[b], C_manual)
    print(f'  Batch {b}: einsum matches A[{b}]@B[{b}]? {match}')

# Batch dot products
v1 = np.random.randn(batch_size, 5)  # 3 vectors of dim 5
v2 = np.random.randn(batch_size, 5)
batch_dots = np.einsum('bi,bi->b', v1, v2)
print(f'\nBatch dot products: {batch_dots.round(3)}')
print(f'  Manual check: {[np.dot(v1[i], v2[i]).round(3) for i in range(batch_size)]}')

# Batch quadratic form: xᵀMx for multiple x vectors
M = np.array([[2, 1], [1, 3]])  # shared matrix
X = np.random.randn(batch_size, 2)  # batch of vectors
quads = np.einsum('bi,ij,bj->b', X, M, X)
print(f'\nBatch quadratic form xᵀMx: {quads.round(3)}')

---

## 🔬 Multi-Head Self-Attention via einsum

The core of every Transformer, expressed purely in einsum:

```
Q: (batch, heads, seq_q, d_k)    'bhid'
K: (batch, heads, seq_k, d_k)    'bhjd'
V: (batch, heads, seq_k, d_v)    'bhjd'

Step 1: scores = QKᵀ/√d_k     → 'bhid,bhjd->bhij'
Step 2: weights = softmax(scores)
Step 3: output = weights @ V   → 'bhij,bhjd->bhid'
```

In [ ]:
# Multi-Head Attention in pure einsum
np.random.seed(42)

batch, heads, seq_len, d_k = 2, 4, 6, 8

Q = np.random.randn(batch, heads, seq_len, d_k)
K = np.random.randn(batch, heads, seq_len, d_k)
V = np.random.randn(batch, heads, seq_len, d_k)

# Step 1: Attention scores — 'bhid,bhjd->bhij'
#   b,h = batch, head (free — kept)
#   i = query position (free)
#   j = key position (free)
#   d = d_k dimension (DUMMY — summed = dot product!)
scores = np.einsum('bhid,bhjd->bhij', Q, K) / np.sqrt(d_k)

print(f'Q shape:      {Q.shape}  (batch, heads, seq, d_k)')
print(f'K shape:      {K.shape}')
print(f'Scores shape: {scores.shape}  (batch, heads, seq_q, seq_k)')

# Step 2: Softmax
def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

attn_weights = softmax(scores, axis=-1)
print(f'\nAttention weights shape: {attn_weights.shape}')
print(f'Weights sum to 1 per query? {np.allclose(attn_weights.sum(axis=-1), 1.0)}')

# Step 3: Weighted sum of values — 'bhij,bhjd->bhid'
#   i = query position (free)
#   j = key position (DUMMY — summed = weighted average!)
#   d = value dimension (free)
output = np.einsum('bhij,bhjd->bhid', attn_weights, V)
print(f'Output shape: {output.shape}  (same as Q — each query gets a result)')

# Show attention pattern for batch 0, head 0
print(f'\nAttention pattern (batch=0, head=0):')
w = attn_weights[0, 0]
print(f'{"":>4}', end='')
for j in range(seq_len):
    print(f'  K{j}', end='')
print()
for i in range(seq_len):
    print(f'Q{i}: ', end='')
    for j in range(seq_len):
        print(f'{w[i,j]:.2f} ', end='')
    print()

print('\n→ This is EXACTLY how GPT/BERT/LLaMA compute attention')
print('→ einsum makes the tensor contractions explicit and readable')

In [ ]:
# ══ REAL PyTorch Self-Attention via einsum ══
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class EinsumAttention(nn.Module):
    """Multi-head attention using ONLY einsum for tensor ops."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x):
        B, S, D = x.shape
        H, dk = self.n_heads, self.d_k
        
        # Project and reshape: (B, S, D) → (B, H, S, dk)
        Q = self.W_Q(x).view(B, S, H, dk).transpose(1, 2)
        K = self.W_K(x).view(B, S, H, dk).transpose(1, 2)
        V = self.W_V(x).view(B, S, H, dk).transpose(1, 2)
        
        # Attention scores via einsum: 'bhid,bhjd->bhij'
        scores = torch.einsum('bhid,bhjd->bhij', Q, K) / math.sqrt(dk)
        attn = F.softmax(scores, dim=-1)
        
        # Weighted values via einsum: 'bhij,bhjd->bhid'
        out = torch.einsum('bhij,bhjd->bhid', attn, V)
        
        # Reshape back: (B, H, S, dk) → (B, S, D)
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

# Test it
d_model, n_heads = 32, 4
attn = EinsumAttention(d_model, n_heads)
x = torch.randn(2, 10, d_model)  # batch=2, seq=10
out = attn(x)

print(f'Input:  {x.shape}  (batch=2, seq=10, d_model={d_model})')
print(f'Output: {out.shape}  (same shape — attention preserves dims)')
print(f'Params: {sum(p.numel() for p in attn.parameters()):,}')
print('\n→ torch.einsum is used in many real Transformer implementations')
print('→ It makes the math transparent: you can READ the tensor contraction')

## 5. Kronecker Delta and Identity

The Kronecker delta δᵢⱼ is the identity matrix in index notation:

$$\delta_{ij} = \begin{cases} 1 & i = j \\ 0 & i \neq j \end{cases}$$

Key property: δᵢⱼaⱼ = aᵢ (selects the i-th component)

In [ ]:
# Kronecker delta = identity matrix

n = 4
delta = np.eye(n)  # δᵢⱼ
print(f'Kronecker delta (δᵢⱼ) = Identity matrix:')
print(delta)

# Property 1: δᵢⱼ aⱼ = aᵢ (selecting component)
a = np.array([10, 20, 30, 40])
result = np.einsum('ij,j->i', delta, a)
print(f'\nδᵢⱼ aⱼ = aᵢ: {result}  (identity transformation)')
print(f'Original a:   {a}')
print(f'Match: {np.array_equal(result, a)}')

# Property 2: δᵢᵢ = n (trace of identity = dimension)
trace_delta = np.einsum('ii->', delta)
print(f'\nδᵢᵢ = Tr(I) = {trace_delta} (equals dimension n={n})')

# Property 3: δᵢⱼ δⱼₖ = δᵢₖ (composition)
delta_compose = np.einsum('ij,jk->ik', delta, delta)
print(f'\nδᵢⱼ δⱼₖ = δᵢₖ? {np.array_equal(delta_compose, delta)}')

# ML usage: Skip connection as identity
x = np.random.randn(4)
f_x = np.random.randn(4) * 0.1  # small perturbation
residual = np.einsum('ij,j->i', delta, x) + f_x  # δᵢⱼxⱼ + f(x)ᵢ = xᵢ + f(x)ᵢ
manual_residual = x + f_x
print(f'\nResidual connection via δᵢⱼ:')
print(f'  δᵢⱼxⱼ + f(x)ᵢ = x + f(x): {np.allclose(residual, manual_residual)}')
print('→ Skip connections ARE the Kronecker delta in disguise!')

## 6. Advanced: Tensor Contraction & Higher-Order Ops

einsum handles operations that have no simple NumPy equivalent:

In [ ]:
# Advanced tensor operations
np.random.seed(42)

# 1. Bilinear form: yₖ = Σᵢⱼ xᵢ Wᵢⱼₖ zⱼ
x = np.random.randn(3)
z = np.random.randn(4)
W = np.random.randn(3, 4, 5)  # bilinear weight tensor
y = np.einsum('i,ijk,j->k', x, W, z)
print(f'=== Bilinear Form: yₖ = xᵢ Wᵢⱼₖ zⱼ ===')
print(f'  x({x.shape}) × W({W.shape}) × z({z.shape}) = y({y.shape})')
print(f'  y = {y.round(3)}')

# 2. Batch outer products
a_batch = np.random.randn(5, 3)  # 5 vectors of dim 3
b_batch = np.random.randn(5, 4)  # 5 vectors of dim 4
outer_batch = np.einsum('bi,bj->bij', a_batch, b_batch)
print(f'\n=== Batch Outer Products ===')
print(f'  a({a_batch.shape}), b({b_batch.shape}) → outer({outer_batch.shape})')

# 3. Tensor trace (contract first and last axes)
T = np.random.randn(3, 4, 3)  # 3-tensor
partial_trace = np.einsum('iji->j', T)
print(f'\n=== Partial Trace: Σᵢ T_iji ===')
print(f'  T({T.shape}) → trace over first/last → result({partial_trace.shape})')

# 4. CP Decomposition reconstruction
R = 3  # rank
a_cp = np.random.randn(4, R)
b_cp = np.random.randn(5, R)
c_cp = np.random.randn(6, R)
T_approx = np.einsum('ir,jr,kr->ijk', a_cp, b_cp, c_cp)
print(f'\n=== CP Decomposition: T_ijk ≈ Σᵣ aᵢᵣ bⱼᵣ cₖᵣ ===')
print(f'  a({a_cp.shape}), b({b_cp.shape}), c({c_cp.shape})')
print(f'  Reconstructed T: {T_approx.shape}')

# 5. Graph attention: message passing
n_nodes, d_feat, d_out = 5, 8, 4
Adj = (np.random.rand(n_nodes, n_nodes) > 0.5).astype(float)  # adjacency
np.fill_diagonal(Adj, 1)  # self-loops
H = np.random.randn(n_nodes, d_feat)  # node features
W_gnn = np.random.randn(d_feat, d_out)  # weight matrix
H_new = np.einsum('ij,jd,dk->ik', Adj, H, W_gnn)
print(f'\n=== GNN Message Passing: H_new = A × H × W ===')
print(f'  A({Adj.shape}) × H({H.shape}) × W({W_gnn.shape}) → H_new({H_new.shape})')
print('  This is one GNN layer in a single einsum call!')

## 7. einsum Performance & Optimization

In [ ]:
# Performance comparison: einsum vs dedicated functions
import time
np.random.seed(42)

n = 500
A = np.random.randn(n, n)
B = np.random.randn(n, n)

# Warm up
_ = np.einsum('ik,kj->ij', A, B)
_ = A @ B

# Benchmark
trials = 20

t0 = time.perf_counter()
for _ in range(trials):
    C1 = np.einsum('ik,kj->ij', A, B)
t_einsum = (time.perf_counter() - t0) / trials

t0 = time.perf_counter()
for _ in range(trials):
    C2 = np.einsum('ik,kj->ij', A, B, optimize=True)
t_einsum_opt = (time.perf_counter() - t0) / trials

t0 = time.perf_counter()
for _ in range(trials):
    C3 = A @ B
t_matmul = (time.perf_counter() - t0) / trials

print(f'Matrix multiply ({n}×{n}):')
print(f'  einsum:              {t_einsum*1000:.2f} ms')
print(f'  einsum (optimize):   {t_einsum_opt*1000:.2f} ms')
print(f'  A @ B (BLAS):        {t_matmul*1000:.2f} ms')
print(f'\n→ For simple matmul, A @ B is faster (uses optimized BLAS)')
print(f'→ Use einsum when you need complex contractions, batching, or readability')

# Where einsum SHINES: complex multi-tensor contractions
print(f'\n--- Where einsum shines ---')
Q = np.random.randn(8, 4, 32, 64)  # (batch, heads, seq, d_k)
K = np.random.randn(8, 4, 32, 64)

t0 = time.perf_counter()
for _ in range(trials):
    s1 = np.einsum('bhid,bhjd->bhij', Q, K)
t1 = (time.perf_counter() - t0) / trials

t0 = time.perf_counter()
for _ in range(trials):
    s2 = np.matmul(Q, K.transpose(0, 1, 3, 2))
t2 = (time.perf_counter() - t0) / trials

print(f'Multi-head attention scores Q({Q.shape}) × Kᵀ:')
print(f'  einsum:    {t1*1000:.2f} ms')
print(f'  transpose+matmul: {t2*1000:.2f} ms')
print(f'  Results match: {np.allclose(s1, s2)}')
print(f'→ einsum is clean: no manual transpose needed!')

---

## 🔬 Contraction Order & Computational Cost

For multi-tensor contractions, **order matters dramatically**.

For A(m×n) × B(n×p) × C(p×q):
- (AB)C costs O(mnp + mpq)
- A(BC) costs O(npq + mnq)

With large shape differences, one order can be 100× faster.

In [ ]:
# Contraction order matters!
np.random.seed(42)

# Extreme shape: A(1000×10) × B(10×1000) × C(1000×10)
m, n, p, q = 1000, 10, 1000, 10
A = np.random.randn(m, n)
B = np.random.randn(n, p)
C = np.random.randn(p, q)

# Order 1: (AB)C
t0 = time.perf_counter()
for _ in range(50):
    r1 = (A @ B) @ C
t_order1 = (time.perf_counter() - t0) / 50

# Order 2: A(BC)
t0 = time.perf_counter()
for _ in range(50):
    r2 = A @ (B @ C)
t_order2 = (time.perf_counter() - t0) / 50

print(f'A({m}×{n}) × B({n}×{p}) × C({p}×{q})')
print(f'  (AB)C: {t_order1*1000:.3f} ms   cost = mnp + mpq = {m*n*p + m*p*q:,}')
print(f'  A(BC): {t_order2*1000:.3f} ms   cost = npq + mnq = {n*p*q + m*n*q:,}')
print(f'  Speedup: {t_order1/t_order2:.1f}×')
print(f'  Results match: {np.allclose(r1, r2)}')

# einsum with optimize=True finds the best order automatically
t0 = time.perf_counter()
for _ in range(50):
    r3 = np.einsum('ij,jk,kq->iq', A, B, C, optimize=True)
t_opt = (time.perf_counter() - t0) / 50

print(f'\n  einsum(optimize=True): {t_opt*1000:.3f} ms')
print(f'  Results match: {np.allclose(r1, r3)}')
print('\n→ optimize=True lets einsum find the cheapest contraction order')
print('→ The opt_einsum package does this even better for complex cases')

## 8. Complete einsum Cheat Sheet

In [ ]:
# Complete einsum cheat sheet — verify every operation
np.random.seed(42)

# Vectors
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# Matrices
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
S = np.random.randn(3, 3)  # square

# Batched
Ab = np.random.randn(2, 3, 4)
Bb = np.random.randn(2, 4, 5)

cheat_sheet = [
    # (description, einsum_string, operands, verification_func)
    ('Vector sum',          'i->',      [a],     lambda: np.sum(a)),
    ('Dot product',         'i,i->',    [a, b],  lambda: np.dot(a, b)),
    ('Element-wise ×',      'i,i->i',   [a, b],  lambda: a * b),
    ('Outer product',       'i,j->ij',  [a, b],  lambda: np.outer(a, b)),
    ('Matrix multiply',     'ik,kj->ij',[A, B],  lambda: A @ B),
    ('Matrix-vector',       'ij,j->i',  [S, a],  lambda: S @ a),
    ('Transpose',           'ij->ji',   [A],     lambda: A.T),
    ('Trace',               'ii->',     [S],     lambda: np.trace(S)),
    ('Diagonal',            'ii->i',    [S],     lambda: np.diag(S)),
    ('Frobenius ‖A‖²',     'ij,ij->',  [S, S],  lambda: np.sum(S**2)),
    ('Batch matmul',        'bik,bkj->bij', [Ab, Bb], lambda: np.matmul(Ab, Bb)),
]

print(f'{"Operation":<20} {"einsum":<20} {"Shape":<15} {"Match?"}')
print('─' * 65)
for desc, subscripts, ops, verify in cheat_sheet:
    result = np.einsum(subscripts, *ops)
    expected = verify()
    match = np.allclose(result, expected)
    shape = str(result.shape) if result.ndim > 0 else 'scalar'
    print(f'{desc:<20} {subscripts:<20} {shape:<15} {"✓" if match else "✗"}')

print('\n→ einsum is a UNIVERSAL tensor operation language')
print('→ Every operation above has ONE consistent syntax')

## Summary

### Core Rules:
| Rule | Meaning |
|---|---|
| Index appears **twice** | Summed over (dummy/contracted) |
| Index appears **once** | Free — labels output component |
| Index in output (`->`) | Kept in result |
| Index NOT in output | Summed away |

### Key Operations:
| Operation | einsum | ML Use |
|---|---|---|
| Dot product | `'i,i->'` | Neuron computation |
| Matmul | `'ik,kj->ij'` | Linear layers |
| Batch matmul | `'bik,bkj->bij'` | Batch processing |
| Attention scores | `'bhid,bhjd->bhij'` | Transformer core |
| Attention output | `'bhij,bhjd->bhid'` | Value aggregation |

### Key Takeaways:
1. **Einstein convention** = repeated indices are summed
2. **einsum** = one function for ALL tensor operations
3. **The output string** controls what gets summed vs kept
4. **Batch dimensions** are trivial — just add a batch index
5. **Use `optimize=True`** for multi-tensor contractions